# Prompt Optimization with Tool Calling
> **Reference Tutorial:** This notebook accompanies the *Prompt Optimization with Tool Calling* tutorial on SAP Developers.  
> Each section below maps directly to a step in the tutorial. Run the cells in order from top to bottom.

---

## Overview

This notebook demonstrates an end-to-end prompt optimization pipeline for tool calling on SAP AI Core using a BFCL v3 dataset. The pipeline:

1. Loads and normalizes a BFCL v3 parallel-multiple dataset
2. Splits it into train and test sets
3. Uploads all files to AI Core's built-in dataset storage
4. Registers a dataset artifact
5. Pushes a base prompt template to the Prompt Registry
6. Creates an optimization configuration targeting Gemini 2.5 Pro with GPT-4o as the reference model
7. Runs the optimization execution using the `JSON_Match` metric
8. Monitors progress until `COMPLETED`
9. Retrieves and inspects the optimized prompt
10. Compares base vs optimized prompt via live inference

> ⚠️ **Prerequisites:** Ensure your `.env` file is configured with `AICORE_BASE_URL`, `AICORE_AUTH_URL`, `AICORE_CLIENT_ID`, `AICORE_CLIENT_SECRET`, and `AICORE_RESOURCE_GROUP` before running this notebook.

---

## Step 1 — Environment Variables Setup & Connect to AI Core

**What this step does:**  
Loads credentials from the `.env` file and initializes the `GenAIHubProxyClient` — the main entry point for all AI Core API calls in Python.

**Create a `.env` file** in the same directory as this notebook with the following content:

```env
AICORE_CLIENT_ID=<your client id>
AICORE_CLIENT_SECRET=<your client secret>
AICORE_AUTH_URL=<your auth url>
AICORE_BASE_URL=<your base url>
AICORE_RESOURCE_GROUP=<your resource group>
```

> 💡 No S3 or AWS credentials are required for this flow — all files are uploaded directly to AI Core's built-in dataset storage via the `/lm/dataset/files` endpoint.

In [2]:
from collections import defaultdict
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient
from dotenv import load_dotenv
import os
import json
import requests
import random
from urllib.parse import quote
from pathlib import Path
from typing import List, Tuple
import time
from ai_api_client_sdk.models.parameter_binding import ParameterBinding
from ai_api_client_sdk.models.input_artifact_binding import InputArtifactBinding
from pydantic import BaseModel
from ai_api_client_sdk.models.artifact import Artifact

load_dotenv(override=True)

# ── SAP AI Core client ────────────────────────────────────────────────────────
client = GenAIHubProxyClient(
    base_url=os.getenv("AICORE_BASE_URL"),
    auth_url=os.getenv("AICORE_AUTH_URL"),
    client_id=os.getenv("AICORE_CLIENT_ID"),
    client_secret=os.getenv("AICORE_CLIENT_SECRET"),
    resource_group=os.getenv("AICORE_RESOURCE_GROUP")
)
resource_group = client.request_header[
    client.ai_core_client.rest_client.resource_group_header
]

print("✅ Connected to AI Core")
print(f"   Resource group: {resource_group}")

✅ Connected to AI Core
   Resource group: grounding


---

## Step 2 — Configure Optimization Parameters

**What this step does:**  
Defines all configuration constants used throughout the notebook — dataset path, prompt name/version, reference model, target model, metric, and the Pydantic models for the prompt template spec.

**Key parameters:**
| Parameter | Value | Description |
|---|---|---|
| `REFERENCE_MODEL` | `gpt-4o:2024-08-06` | Teacher model used for evaluation |
| `TARGET_MODELS` | `gemini-2.5-pro:001` | Model to optimize the prompt for |
| `METRIC` | `JSON_Match` | Evaluates whether tool call JSON matches the golden answer |
| `N_TRAIN_SAMPLES` | 25 | Number of samples used to train/refine the prompt |
| `N_TEST_SAMPLES` | 15 | Number of samples used to evaluate candidate prompts |

> ⚠️ Verify that `gpt-4o:2024-08-06` and `gemini-2.5-pro:001` are available in your AI Core tenant before running. Check via Generative AI Hub → Models.

In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
BFCL_DATASET      = "BFCL_v3_parallel_multiple_10tools.json"
BFCL_DATASET_MODE = "parallel_multiple"
N_TRAIN_SAMPLES   = 25
N_TEST_SAMPLES    = 15

PROMPT_NAME    = "bfcl-tool-base"
PROMPT_VERSION = "0.0.1"
SCENARIO       = "genai-optimizations"

SYSTEM_PROMPT   = "You are a helpful assistant."
PROMPT_TEMPLATE = "{{?question}}"
FIELDS          = ["question"]

# Use a model confirmed available in your region
REFERENCE_MODEL = "gpt-4o:2024-08-06"
TARGET_MODELS = {
    "gemini-2.5-pro:001": "bfcl-tool-optimized-gemini25:0.0.1",
}
METRIC = "JSON_Match"

# ── Pydantic models ───────────────────────────────────────────────────────────
class PromptTemplateMsg(BaseModel):
    role: str
    content: str

class PromptTemplateSpec(BaseModel):
    template: List[PromptTemplateMsg]

prompt = PromptTemplateSpec(template=[
    PromptTemplateMsg(role="system", content=SYSTEM_PROMPT),
    PromptTemplateMsg(role="user",   content=PROMPT_TEMPLATE),
])

print("✅ Configuration set")
print(f"   Dataset       : {BFCL_DATASET}")
print(f"   Scenario      : {SCENARIO}")
print(f"   Prompt name   : {PROMPT_NAME}:{PROMPT_VERSION}")
print(f"   Reference     : {REFERENCE_MODEL}")
print(f"   Target        : {list(TARGET_MODELS.keys())}")
print(f"   Metric        : {METRIC}")

✅ Configuration set
   Dataset       : BFCL_v3_parallel_multiple_10tools.json
   Scenario      : genai-optimizations
   Prompt name   : bfcl-tool-base:0.0.1
   Reference     : gpt-4o:2024-08-06
   Target        : ['gemini-2.5-pro:001']
   Metric        : JSON_Match


---

## Step 3 — Load and Normalize the BFCL v3 Dataset

**What this step does:**  
Loads the BFCL v3 dataset file and normalizes it into the SAP optimizer golden format. This includes:

- **`read_bfcl_file`** — a robust reader that handles 3 BFCL file formats: JSON array, standard JSONL, and concatenated JSON objects (the native BFCL v3 format).
- **`normalize_bfcl_tool`** — converts raw BFCL tool definitions to OpenAI ChatCompletions format (e.g., `float` → `number`, `dict` → `object`, `any` → `string`).
- **`dedupe_tool_name`** / **`union_bfcl_tools`** — deduplicates tools with identical names but different schemas across samples.
- **`detect_tool_key`** / **`detect_question_key`** — auto-detects the correct field names in the dataset.
- **`build_golden`** — converts each BFCL sample into a SAP optimizer golden record:
  - `fields.question` → the user query text
  - `answer` → a JSON object string of merged tool calls (e.g. `{"weather_forecast": {...}, "calculate_distance": {...}}`)

**Output format per golden record:**
```json
{
  "fields": { "question": "I'm planning a trip to Japan..." },
  "answer": "{"currency_conversion": {"amount": [5000.0], ...}, "calculate_distance": {...}}"
}
```

> 💡 The `answer` must be a JSON **object** string (not an array), where each key is a tool name and each value is its arguments dict.

In [4]:
# ── BFCL v3 file reader ───────────────────────────────────────────────────────
def read_bfcl_file(file_path: Path) -> list:
    """Robust reader for BFCL v3 files (concatenated JSON objects)."""
    with open(file_path, "r") as f:
        content = f.read().strip()

    print(f"File size: {len(content):,} bytes")

    # Format 1: JSON array [ {...}, {...} ]
    if content.startswith("["):
        try:
            result = json.loads(content)
            if result and isinstance(result[0], str):
                print("Detected double-encoded strings — decoding...")
                result = [json.loads(item) for item in result]
            print(f"Loaded as JSON array: {len(result)} records")
            return result
        except json.JSONDecodeError:
            pass

    # Format 2: Standard JSONL — one complete object per line
    if "\n" in content:
        objects = []
        for line in content.split("\n"):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    objects.append(obj)
            except json.JSONDecodeError:
                pass
        if objects:
            print(f"Loaded as JSONL: {len(objects)} records")
            return objects

    # Format 3: Concatenated/multi-line JSON objects — use raw_decode to scan through
    # This is the correct format for BFCL v3: {...}\n{...}\n{...}
    objects = []
    decoder = json.JSONDecoder()
    idx = 0
    while idx < len(content):
        while idx < len(content) and content[idx] in " \t\n\r":
            idx += 1
        if idx >= len(content):
            break
        if content[idx] != "{":
            idx += 1
            continue
        try:
            obj, end_idx = decoder.raw_decode(content, idx)
            if isinstance(obj, dict):
                objects.append(obj)
            idx = end_idx
        except json.JSONDecodeError:
            idx += 1
            continue

    print(f"Loaded as concatenated JSON: {len(objects)} records")
    return objects


In [5]:
# ── BFCL v3 normalisation ─────────────────────────────────────────────────────
def reformat_tool_name(name: str) -> str:
    return name.replace(".", "_")

def normalize_bfcl_tool(tool: dict) -> dict:
    """Convert raw BFCL tool definition to OpenAI ChatCompletions format."""
    tool = json.loads(json.dumps(tool))       # deep copy
    tool["parameters"]["type"] = "object"     # BFCL uses 'dict'
    tool["name"] = reformat_tool_name(tool["name"])
    for param in tool["parameters"].get("properties", {}).values():
        if param["type"] == "float":
            param["type"] = "number"
        elif param["type"] == "dict":
            param["type"] = "object"
        elif param["type"] == "any":
            param["type"] = "string"
        elif param["type"] == "array" and "items" in param:
            if param["items"].get("type") == "float":
                param["items"]["type"] = "number"
            elif param["items"].get("type") == "dict":
                param["items"]["type"] = "object"
    return {"type": "function", "function": tool}



In [6]:
def dedupe_tool_name(tool_name: str, answers: list, occurrences: int):
    new_name = f"{tool_name}_n{occurrences}"
    new_answers = []
    for answer in answers:
        new_answer = {}
        for k, v in answer.items():
            new_answer[reformat_tool_name(new_name if k == tool_name else k)] = v
        new_answers.append(new_answer)
    return new_name, new_answers

def union_bfcl_tools(samples: list, tool_key: str = "function") -> list:
    """Union and deduplicate all tools across samples into one normalized list."""
    tool_map = {}
    tool_name_occurrences = defaultdict(int)
    tool_name_to_definition = {}
    for sample in samples:
        for tool in sample[tool_key]:
            tool_name = tool["name"]
            tool_name_occurrences[tool_name] += 1
            is_duplicate = (
                tool_name in tool_name_to_definition and
                tool != tool_name_to_definition[tool_name]
            )
            if is_duplicate:
                new_name, new_answers = dedupe_tool_name(
                    tool_name, sample["answer"], tool_name_occurrences[tool_name]
                )
                print(f"WARNING: duplicate tool {tool_name!r} → renamed to {new_name!r}")
                tool["name"]     = new_name
                sample["answer"] = new_answers
            tool_name_to_definition[tool["name"]] = tool
            normalized = normalize_bfcl_tool(tool)
            tool_map[normalized["function"]["name"]] = normalized
    return list(tool_map.values())

def detect_tool_key(sample: dict) -> str:
    """Detect which key holds the tool definitions in a BFCL sample."""
    for candidate in ["function", "functions", "tools", "tool"]:
        if candidate in sample:
            return candidate
    raise KeyError(
        f"Cannot find tool key in sample. Available keys: {list(sample.keys())}"
    )

def detect_question_key(sample: dict) -> str:
    """Detect which key holds the question/messages in a BFCL sample."""
    for candidate in ["question", "messages", "turns", "prompt"]:
        if candidate in sample:
            return candidate
    raise KeyError(
        f"Cannot find question key in sample. Available keys: {list(sample.keys())}"
    )



In [7]:
def build_golden(sample: dict, question_key: str = "question") -> dict:
    """Convert one BFCL sample to a SAP optimizer golden record."""
    raw_question = sample[question_key]
    if isinstance(raw_question[0], list):
        question = "\n".join(q["content"] for q in raw_question[0])
    elif isinstance(raw_question[0], dict):
        question = "\n".join(q["content"] for q in raw_question)
    else:
        question = str(raw_question)

    # Merge all tool calls into a single flat dict
    # e.g. [{"weather": {...}}, {"hotel": {...}}] → {"weather": {...}, "hotel": {...}}
    merged_answer = {}
    for answer in sample["answer"]:
        for key, value in answer.items():
            key = reformat_tool_name(key)
            # Flatten nested single-element arrays (BFCL quirk)
            if isinstance(value, list) and len(value) == 1 and isinstance(value[0], list):
                value = value[0]
            # Float → string to match normalized tool schema
            if isinstance(value, float):
                value = str(value)
            elif isinstance(value, list):
                value = [str(v) if isinstance(v, float) else v for v in value]
            merged_answer[key] = value

    # answer must be a JSON object string, not a JSON array string
    return {
        "fields": {"question": question},
        "answer": json.dumps(merged_answer)   # "{...}" not "[{...}]"
    }

def load_bfcl_dataset(
    dataset_path: Path,
    n_train: int,
    n_test: int,
) -> Tuple[list, list, list]:
    """Load, sample, normalise, and split BFCL v3 data."""
    samples = read_bfcl_file(dataset_path)
    print(f"Total records in file: {len(samples)}")

    if samples:
        print(f"Sample keys: {list(samples[0].keys())}")

    n_total = min(n_train + n_test, len(samples))
    random.seed(42)
    sampled = random.sample(samples, n_total)

    tool_key     = detect_tool_key(sampled[0])
    question_key = detect_question_key(sampled[0])
    print(f"Tool key: '{tool_key}' | Question key: '{question_key}'")

    tools         = union_bfcl_tools(sampled, tool_key=tool_key)
    train_goldens = [build_golden(s, question_key=question_key) for s in sampled[:n_train]]
    test_goldens  = [build_golden(s, question_key=question_key) for s in sampled[n_train:]]
    return train_goldens, test_goldens, tools

# ── Load dataset ──────────────────────────────────────────────────────────────
dataset_path = Path(BFCL_DATASET)
train_goldens, test_goldens, tools = load_bfcl_dataset(
    dataset_path, N_TRAIN_SAMPLES, N_TEST_SAMPLES
)
print(f"Train goldens : {len(train_goldens)}")
print(f"Test goldens  : {len(test_goldens)}")
print(f"Unioned tools : {len(tools)}")
print(f"\nSample golden:\n{json.dumps(train_goldens[0], indent=2)}")

File size: 428,329 bytes
Loaded as concatenated JSON: 35 records
Total records in file: 35
Sample keys: ['id', 'question', 'function', 'answer']
Tool key: 'function' | Question key: 'question'
Train goldens : 25
Test goldens  : 10
Unioned tools : 10

Sample golden:
{
  "fields": {
    "question": "I'm planning a trip to Japan. I have 5000 US dollars and want to know how much that is in Japanese Yen. I'd also like to know the distance from Tokyo to Kyoto in kilometers. And while I'm researching Japanese companies, could you get me the latest stock price for Toyota?"
  },
  "answer": "{\"currency_conversion\": {\"amount\": [5000.0], \"from_currency\": [\"USD\", \"US Dollars\", \"US Dollar\"], \"to_currency\": [\"JPY\", \"Japanese Yen\"]}, \"calculate_distance\": {\"origin\": [\"Tokyo\"], \"destination\": [\"Kyoto\"], \"unit\": [\"km\", \"\"]}, \"get_stock_info\": {\"company\": [\"Toyota\", \"TM\"], \"metric\": [\"price\"]}}"
}


---

## Step 4 — Upload Dataset Files to AI Core Storage

**What this step does:**  
Serializes the four prepared objects to local JSON files, then uploads each one to a shared folder in AI Core's built-in dataset storage using the `/lm/dataset/files` endpoint.

**Files uploaded:**
| File | Contents |
|---|---|
| `bfcl_train.json` | 25 golden records used to train/refine the prompt |
| `bfcl_test.json` | 15 golden records used to evaluate candidate prompts |
| `bfcl_tools.json` | Union of all normalized tool definitions across samples |
| `bfcl_prompt_template.json` | The base prompt template spec |

All four files land in the same remote folder: `default/datasets/bfcl-optimizer/`

After uploading, the shared folder is registered as a single **dataset artifact** under the `genai-optimizations` scenario. The optimizer reads all files from this artifact folder.

> 💡 `get_or_create_artifact` checks for an existing artifact at the same URL before creating a new one — safe to re-run without creating duplicates.

In [8]:
# ── Serialize files for upload ────────────────────────────────────────────────
train_local  = "./bfcl_train.json"
test_local   = "./bfcl_test.json"
tools_local  = "./bfcl_tools.json"
prompt_local = "./bfcl_prompt_template.json"

with open(train_local,  "w") as f: json.dump(train_goldens,    f, indent=2)
with open(test_local,   "w") as f: json.dump(test_goldens,     f, indent=2)
with open(tools_local,  "w") as f: json.dump(tools,            f, indent=2)
with open(prompt_local, "w") as f: json.dump(prompt.model_dump(), f, indent=2)
print("Local files written.")



Local files written.


In [9]:
# ── Helpers: upload & artifact ────────────────────────────────────────────────
def upload_file(local_path: str, remote_subfolder: str, filename: str) -> str:
    """Upload file to AI Core dataset storage. Returns folder path 'default/<subfolder>'."""
    full_path    = f"default/{remote_subfolder}/{filename}"
    encoded_path = quote(full_path, safe="")
    url          = f"{client.ai_core_client.base_url}/lm/dataset/files/{encoded_path}"
    headers      = {**client.request_header, "Content-Type": "application/json"}
    with open(local_path, "rb") as f:
        res = requests.put(url, params={"overwrite": "true"}, headers=headers, data=f)
    print(f"  Upload [{filename}]: {res.status_code}")
    res.raise_for_status()
    return f"default/{remote_subfolder}"



In [10]:
def get_or_create_artifact(name: str, folder_path: str, description: str) -> str:
    """Register folder as artifact. Returns artifact_id."""
    artifact_url = f"ai://{folder_path}"
    existing = client.ai_core_client.artifact.query(
        resource_group=resource_group, scenario_id=SCENARIO
    )
    for art in existing.resources:
        if art.url == artifact_url:
            print(f"  Reusing artifact [{name}]: {art.id}")
            return art.id
    resp = client.ai_core_client.artifact.create(
        name=name, kind=Artifact.Kind.DATASET,
        url=artifact_url, scenario_id=SCENARIO,
        resource_group=resource_group, description=description
    )
    print(f"  Created artifact [{name}]: {resp.id}")
    return resp.id

# ── Upload all files to shared folder ────────────────────────────────────────
REMOTE_SUBFOLDER = "datasets/bfcl-optimizer"
print("Uploading files...")
shared_folder = upload_file(train_local,  REMOTE_SUBFOLDER, "bfcl_train.json")
upload_file(test_local,   REMOTE_SUBFOLDER, "bfcl_test.json")
upload_file(tools_local,  REMOTE_SUBFOLDER, "bfcl_tools.json")
upload_file(prompt_local, REMOTE_SUBFOLDER, "bfcl_prompt_template.json")
print(f"Shared folder: {shared_folder}")

# ── Register dataset artifact ─────────────────────────────────────────────────
optimizer_artifact_id = get_or_create_artifact(
    name="bfcl-optimizer-data",
    folder_path=shared_folder,
    description="BFCL train/test goldens, tools, and prompt template"
)
print(f"Artifact ID: {optimizer_artifact_id}")


Uploading files...
  Upload [bfcl_train.json]: 201
  Upload [bfcl_test.json]: 201
  Upload [bfcl_tools.json]: 201
  Upload [bfcl_prompt_template.json]: 201
Shared folder: default/datasets/bfcl-optimizer
  Reusing artifact [bfcl-optimizer-data]: 49b56062-0824-4124-b008-ddc5e6e075a5
Artifact ID: 49b56062-0824-4124-b008-ddc5e6e075a5


---

## Step 5 — Create and Register the Base Prompt Template

**What this step does:**  
Pushes the base prompt template to the Prompt Registry under the `genai-optimizations` scenario.

The base prompt is intentionally minimal:
- **System:** `"You are a helpful assistant."`
- **User:** `{{?question}}`

The optimizer takes this as its starting point and iteratively rewrites it during execution. When done, the final refined prompt is saved back to the registry under the name specified in `targetPromptMapping` (e.g., `bfcl-tool-optimized-gemini25:0.0.1`).

> 💡 A `409` response means the prompt already exists — this is safe and the existing version is reused automatically.

In [11]:
# ── Push prompt to registry ───────────────────────────────────────────────────
def push_prompt(spec: PromptTemplateSpec, name: str, version: str, scenario: str):
    url  = f"{client.ai_core_client.base_url}/lm/promptTemplates"
    body = {"name": name, "version": version, "scenario": scenario, "spec": spec.model_dump()}
    res  = requests.post(
        url,
        headers={**client.request_header, "Content-Type": "application/json"},
        json=body
    )
    print(f"Prompt registry: {res.status_code} — {res.json().get('message', '')}")
    if res.status_code == 409:
        print("Prompt already exists — reusing.")
        return {"name": name, "version": version}
    res.raise_for_status()
    return res.json()

push_prompt(prompt, PROMPT_NAME, PROMPT_VERSION, SCENARIO)

Prompt registry: 200 — Prompt updated successfully.


{'message': 'Prompt updated successfully.',
 'id': '9a8f46c9-630a-4950-b098-b07c8bd9868d',
 'scenario': 'genai-optimizations',
 'name': 'bfcl-tool-base',
 'version': '0.0.1'}

---

## Step 6 — Register an Optimization Configuration

**What this step does:**  
Creates the optimization configuration that links all inputs together — the artifact, base prompt, reference model, target model, and metric — into one executable setup.

**15 parameter bindings explained:**

| Parameter | Value | Purpose |
|---|---|---|
| `optimizationMetric` | `JSON_Match` | Evaluates tool call JSON accuracy |
| `basePrompt` | `genai-optimizations/bfcl-tool-base:0.0.1` | Starting prompt in the registry |
| `baseModel` | `gpt-4o:2024-08-06` | Reference/teacher model |
| `targetModels` | `gemini-2.5-pro:001` | Model to optimize for |
| `targetPromptMapping` | `gemini-2.5-pro:001=bfcl-tool-optimized-gemini25:0.0.1` | Output prompt name |
| `trainDataset` | `bfcl_train.json` | Training file in the artifact folder |
| `testDataset` | `bfcl_test.json` | Evaluation file in the artifact folder |
| `maximize` | `true` | Maximize the metric score |
| `includeFewShotExamples` | `false` | No few-shot injection |

> 💡 `create_config` checks for an existing configuration with identical parameters before creating a new one — safe to re-run.

In [12]:
# ── Create configuration ──────────────────────────────────────────────────────
def create_config(
    metric: str,
    reference_model: str,
    targets: dict,
    train_filename: str,
    test_filename: str,
    prompt_artifact_id: str,
    prompt_name: str,
    prompt_version: str,
    scenario: str,
) -> str:
    base_prompt = f"{scenario}/{prompt_name}:{prompt_version}"

    input_parameters = [
        ParameterBinding(key="optimizationMetric",     value=metric),
        ParameterBinding(key="basePrompt",             value=base_prompt),
        ParameterBinding(key="baseModel",              value=reference_model),
        ParameterBinding(key="targetModels",           value=",".join(targets.keys())),
        ParameterBinding(
            key="targetPromptMapping",
            value=",".join(f"{k}={v}" for k, v in targets.items())
        ),
        ParameterBinding(key="trainDataset",           value=train_filename),
        ParameterBinding(key="testDataset",            value=test_filename),
        ParameterBinding(key="maximize",               value="true"),
        ParameterBinding(key="correctnessCutoff",      value="none"),
        ParameterBinding(key="includeFewShotExamples", value="false"),
        ParameterBinding(key="promptTemplateScope",    value="tenant"),
        ParameterBinding(key="prototypeMode",          value="false"),
        ParameterBinding(key="fieldEvaluationMetrics", value="none"),
        ParameterBinding(key="modelParams",            value="none"),
        ParameterBinding(key="customMetricId",         value="none"),
    ]

    input_artifacts = [
        InputArtifactBinding(key="prompt-data", artifact_id=prompt_artifact_id)
    ]

    params_dict = {p.key: p.value for p in input_parameters}

    try:
        existing = client.ai_core_client.configuration.query(
            scenario_id=SCENARIO, resource_group=resource_group
        )
        for conf in existing.resources:
            if {p.key: p.value for p in conf.parameter_bindings} == params_dict:
                print(f"Reusing configuration: {conf.id}")
                return conf.id
    except Exception as e:
        print(f"Could not query configs: {e}")

    resp = client.ai_core_client.configuration.create(
        name="bfcl-tool-config",
        scenario_id=SCENARIO,
        executable_id=SCENARIO,
        resource_group=resource_group,
        parameter_bindings=input_parameters,
        input_artifact_bindings=input_artifacts,
    )
    print(f"Created configuration: {resp.id}")
    return resp.id

configuration_id = create_config(
    metric=METRIC,
    reference_model=REFERENCE_MODEL,
    targets=TARGET_MODELS,
    train_filename="bfcl_train.json",
    test_filename="bfcl_test.json",
    prompt_artifact_id=optimizer_artifact_id,
    prompt_name=PROMPT_NAME,
    prompt_version=PROMPT_VERSION,
    scenario=SCENARIO,
)
print(f"Configuration ID: {configuration_id}")

Reusing configuration: 338b6acd-d9b7-4c07-a5f9-588474b25209
Configuration ID: 338b6acd-d9b7-4c07-a5f9-588474b25209


---

## Step 7 — Run the Prompt Optimization Execution & Monitor Progress

**What this step does:**  
Triggers the optimization job and polls its status every 30 seconds until completion.

**Execution status transitions:**
```
UNKNOWN → RUNNING (progress=1/100) → RUNNING (progress=32/100) → RUNNING (progress=70/100) → COMPLETED (progress=100/100)
```

**Expected duration:** ~20–30 minutes for 25 train + 15 test samples.

**If execution fails (`FAILED` / `DEAD`):**  
The cell automatically fetches and prints the execution logs to help diagnose the issue.

> ⚠️ The cell will keep polling until a terminal status is reached. You can interrupt it with `Kernel → Interrupt` if needed — the execution continues running on AI Core even after interruption.

In [13]:
# ── Execute ───────────────────────────────────────────────────────────────────
execution = client.ai_core_client.execution.create(
    configuration_id=configuration_id,
    resource_group=resource_group,
)
execution_id = execution.id
print(f"Execution ID: {execution_id}")



Execution ID: e0e9f319fec5fb90


In [15]:
TERMINAL_STATES = {"COMPLETED", "FAILED", "DEAD", "STOPPED"}

while True:
    status = client.ai_core_client.execution.get(
        execution_id=execution_id, resource_group=resource_group
    )
    # Safely convert enum to string
    status_str = status.status.value if hasattr(status.status, "value") else str(status.status)
    print(f"[{time.strftime('%H:%M:%S')}] {status_str}", end="")

    if hasattr(status, "status_details") and status.status_details:
        progress = status.status_details.get("progress", "")
        print(f"  progress={progress}", end="")
    print()

    if status_str in TERMINAL_STATES:
        if status_str != "COMPLETED":
            try:
                logs = client.ai_core_client.execution.get_logs(
                    execution_id=execution_id, resource_group=resource_group
                )
                print("── Execution logs ──")
                for log in logs.data:
                    print(log.msg)
            except Exception as e:
                print(f"Could not fetch logs: {e}")
        break

    time.sleep(30)

print(f"\nFinal status: {status_str}")

[00:17:49] COMPLETED  progress=100/100

Final status: COMPLETED


---

## Step 8 — Fetch the Full Optimized Prompt by ID

**What this step does:**  
Retrieves the complete optimized prompt template from the Prompt Registry by its ID.

**Replace `optimized_id`** with the actual ID of `bfcl-tool-optimized-gemini25` found in the listing above.

The optimized prompt will be significantly more detailed than the base `"You are a helpful assistant."` — it will contain:
- A structured-output parser role definition
- Last-of-type selection rules for duplicate function calls
- Strict JSON output constraints (no markdown, no backticks)
- Full tool schemas with normalization rules for all 10 functions
- Reasoning steps and concrete exemplars

In [18]:
url = f"{client.ai_core_client.base_url}/lm/promptTemplates"
res = requests.get(url, headers=client.request_header)
templates = res.json()
for t in templates.get("resources", []):
    print(f"  name={t['name']}  version={t['version']}  id={t['id']}")

  name=multi_task_withRG  version=1.1.3  id=35f5c235-b949-49bb-854e-cca0913086ab
  name=expand_text  version=1.1.2  id=c050ab3a-1653-41d5-8d76-1c2f0a4457fd
  name=multi_task_withRG  version=1.1.2  id=9b502632-8ddc-4d44-bc0c-78044f8ca63b
  name=multi_task  version=1.1.1  id=fca6185e-e340-4781-9d13-00e32f510674
  name=facility-json-template  version=1.0.0  id=27ac3122-9b6a-4baa-a7a2-c3670cea83b2
  name=prompt-registry-eval-demo  version=1.0.0  id=9f47e745-5c87-46df-b917-8e7dc829f47c
  name=evalPromptTemplateConfig-227e9e3  version=1.0.0  id=d4416a6f-8f45-458e-81a2-f3d2f346a26e
  name=evalPromptTemplateConfig-8eb5f38  version=1.0.0  id=87faced1-57b6-4445-aa3f-6294f1a5a8b0
  name=evalPromptTemplateConfig-25152b4  version=1.0.0  id=24dfbaad-9609-4a69-abf9-e3053aa90bf2
  name=evalPromptTemplateConfig-de19a80  version=1.0.0  id=5ac8367e-fc1d-4aed-87ae-cb5166112b1c
  name=evalPromptTemplateConfig-fa00f03  version=1.0.0  id=ef294c8f-ded7-4ddf-a3ec-aa415efc7300
  name=evalPromptTemplateConfig-f5

In [19]:
# ── Fetch optimized prompt template ──────────────────────────────────────────
# Replace with the actual ID of bfcl-tool-optimized-gemini25 from the listing above
optimized_id = "d4416a6f-8f45-458e-81a2-f3d2f346a26e"

url = f"{client.ai_core_client.base_url}/lm/promptTemplates/{optimized_id}"
res = requests.get(url, headers=client.request_header)
print(f"Status: {res.status_code}")
optimized = res.json()
print(json.dumps(optimized, indent=2))

Status: 200
{
  "id": "d4416a6f-8f45-458e-81a2-f3d2f346a26e",
  "name": "evalPromptTemplateConfig-227e9e3",
  "version": "1.0.0",
  "scenario": "genai-evaluations",
  "creationTimestamp": "2026-05-22T05:42:59.492000",
  "managedBy": "imperative",
  "isVersionHead": true,
  "spec": {
    "template": [
      {
        "role": "user",
        "content": "You are a helpful assistant specialized in e-manual topics. Answer the following e-manual questions using the provided context. If the answer is not explicitly available in the context, respond with: `The answer is not available in the provided context.` \n\nRequest: {{?topic}}. \n\nContext: {{?groundingOutput}}"
      }
    ],
    "defaults": {},
    "additionalFields": {}
  }
}


---

## Step 09 — Compare Base vs Optimized Prompt via Live Inference

**What this step does:**  
Runs four test questions through both the base and optimized prompts on Gemini 2.5 Pro and compares the outputs side by side.

**For each question the comparison shows:**
- `BASE` output — what the base prompt (`"You are a helpful assistant."`) produces
- `OPTIMIZED` output — what the optimizer-refined prompt produces
- `COMPARISON` — whether each output is valid JSON and which tools were called
- `VERDICT` — whether the optimization was a WIN, both valid, or both invalid

**Expected result:**  
The base prompt typically returns natural language prose. The optimized prompt returns a strict JSON object of tool calls — confirming a successful optimization WIN.

```
BASE         → invalid JSON ❌ | raw: Of course! I can help with all three...
OPTIMIZED    → valid JSON ✅ | tools called: ['currency_conversion', 'event_finder', 'recipe_search']

📈 VERDICT:
🏆 Optimization WIN — base gave prose, optimized gave structured JSON
```

In [33]:
# Find the running orchestration deployment URL
url = f"{client.ai_core_client.base_url}/lm/deployments"
res = requests.get(url, headers=client.request_header)
for d in res.json().get("resources", []):
    print(f"id={d.get('id')}  scenario={d.get('scenarioId'):30s}  status={d.get('status'):10s}  url={d.get('deploymentUrl')}")

id=ded5987286fdd6ac  scenario=orchestration                   status=RUNNING     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/ded5987286fdd6ac
id=de11bc16755c5bf1  scenario=tabular-orchestration           status=DEAD        url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/de11bc16755c5bf1
id=d4bf99dcade90437  scenario=orchestration                   status=STOPPED     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d4bf99dcade90437
id=df19dafeabb2db6d  scenario=foundation-models               status=RUNNING     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/df19dafeabb2db6d
id=d8914af3d48ccbaf  scenario=foundation-models               status=RUNNING     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d8914af3d48ccbaf
id=d1f817e757a42343  scenario=found

In [35]:
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.service import OrchestrationService

# ── Step 1: Fetch both prompt templates from the registry ─────────────────────
def get_prompt_template(template_id: str) -> dict:
    url = f"{client.ai_core_client.base_url}/lm/promptTemplates/{template_id}"
    res = requests.get(url, headers=client.request_header)
    res.raise_for_status()
    return res.json()

def extract_messages(template: dict) -> dict:
    messages = {}
    for msg in template.get("spec", {}).get("template", []):
        messages[msg["role"]] = msg["content"]
    return messages

# Look up IDs by name from the registry
url = f"{client.ai_core_client.base_url}/lm/promptTemplates"
res = requests.get(url, headers=client.request_header)
all_templates = {
    f"{t['name']}:{t['version']}": t["id"]
    for t in res.json().get("resources", [])
}

base_id      = all_templates.get("bfcl-tool-base:0.0.1")
optimized_id = all_templates.get("bfcl-tool-optimized-gemini25:0.0.1")

base_template      = get_prompt_template(base_id)
optimized_template = get_prompt_template(optimized_id)

base_messages      = extract_messages(base_template)
optimized_messages = extract_messages(optimized_template)

print("✅ Loaded base and optimized prompt templates.")
print(f"Base system prompt      : {base_messages['system'][:80]}...")
print(f"Optimized system prompt : {optimized_messages['system'][:80]}...")


# ── Step 2: run_inference using OrchestrationService ────────────────────────
# Paste your orchestration deployment URL here
ORCHESTRATION_DEPLOYMENT_URL = "https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/ded5987286fdd6ac"

def run_inference(system_prompt: str, user_template: str, question: str, model_name: str) -> str:
    user_content = user_template.replace("{{?question}}", question)

    config = OrchestrationConfig(
        llm=LLM(name=model_name),
        template=Template(messages=[
            SystemMessage(system_prompt),
            UserMessage(user_content),
        ]),
    )

    service  = OrchestrationService(api_url=ORCHESTRATION_DEPLOYMENT_URL, config=config)
    response = service.run()
    return response.module_results.llm.choices[0].message.content
# ── Step 3: Compare prompts ───────────────────────────────────────────────────
def clean_json_output(text: str) -> str:
    """Strip markdown code fences that models sometimes wrap around JSON."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()
    return text


def compare_prompts(question: str, model_name: str = "gemini-2.5-pro"):
    print("\n" + "=" * 70)
    print(f"QUESTION:\n{question}")
    print("=" * 70)

    print("\n📌 BASE PROMPT OUTPUT:")
    print("-" * 40)
    try:
        base_output = run_inference(
            system_prompt=base_messages["system"],
            user_template=base_messages["user"],
            question=question,
            model_name=model_name,
        )
        print(base_output)
    except Exception as e:
        base_output = f"ERROR: {e}"
        print(base_output)

    print("\n✅ OPTIMIZED PROMPT OUTPUT:")
    print("-" * 40)
    try:
        optimized_output = run_inference(
            system_prompt=optimized_messages["system"],
            user_template=optimized_messages["user"],
            question=question,
            model_name=model_name,
        )
        print(optimized_output)
    except Exception as e:
        optimized_output = f"ERROR: {e}"
        print(optimized_output)

    print("\n📊 COMPARISON:")
    print("-" * 40)
    for label, output in [("BASE", base_output), ("OPTIMIZED", optimized_output)]:
        cleaned = clean_json_output(output)
        try:
            parsed = json.loads(cleaned)
            tools  = list(parsed.keys())
            print(f"{label:12s} → valid JSON ✅ | tools called: {tools}")
        except json.JSONDecodeError:
            print(f"{label:12s} → invalid JSON ❌ | raw: {output[:120]}")

    print("\n📈 VERDICT:")
    print("-" * 40)
    base_valid      = True
    optimized_valid = True
    try:
        json.loads(clean_json_output(base_output))
    except Exception:
        base_valid = False
    try:
        json.loads(clean_json_output(optimized_output))
    except Exception:
        optimized_valid = False

    if not base_valid and optimized_valid:
        print("🏆 Optimization WIN — base gave prose, optimized gave structured JSON")
    elif base_valid and optimized_valid:
        print("✅ Both valid JSON — compare tool accuracy above")
    elif base_valid and not optimized_valid:
        print("⚠️  Base was valid but optimized was not — check prompt")
    else:
        print("❌ Both invalid — check model or deployment")

    return base_output, optimized_output


# ── Step 4: Run all comparisons ───────────────────────────────────────────────
compare_prompts("What is the weather in Tokyo for the next 3 days in celsius?")

compare_prompts(
    "I have 2000 euros and want to know how much that is in USD. "
    "Also find me a mid-range Italian restaurant in Milan. "
    "And what is Apple's current stock price?"
)

compare_prompts(
    "Book a hotel in Paris for 2 guests from 2025-08-01 to 2025-08-05 "
    "in a deluxe room. Also check the weather in Paris for the next 7 days "
    "in celsius. And find me the distance from Paris to Lyon in km."
)

compare_prompts(
    "Convert 5000 US dollars to Japanese yen. "
    "Find concerts in New York tomorrow. "
    "Search for vegan pasta recipes under 30 minutes."
)

✅ Loaded base and optimized prompt templates.
Base system prompt      : You are a helpful assistant....
Optimized system prompt : You are a structured-output extraction engine that converts user queries into a ...

QUESTION:
What is the weather in Tokyo for the next 3 days in celsius?

📌 BASE PROMPT OUTPUT:
----------------------------------------
As an AI, I cannot provide real-time information like a live weather forecast. My knowledge is not updated in real time.

To get the most accurate and up-to-date 3-day forecast for Tokyo in Celsius, I highly recommend you check one of these reliable sources:

*   **Google Search:** Simply search for "**weather in Tokyo**"
*   **A dedicated weather website:** Such as AccuWeather, The Weather Channel, or BBC Weather.

A typical forecast from one of these services will look something like this (this is an **example only**):

*   **Today:** Partly cloudy. High: 25°C, Low: 18°C.
*   **Tomorrow:** Light rain showers. High: 22°C, Low: 17°C.
*   **Th

('Of course! Here is the information for your three requests.\n\n### 1. Currency Conversion: USD to JPY\n\nAs of my latest update, **5000 US Dollars** is approximately:\n\n**¥785,500 Japanese Yen**\n\n*Please note: Exchange rates fluctuate constantly. This conversion is based on the current mid-market rate. The actual rate you get from a bank or currency exchange service will be slightly different due to their fees.*\n\n---\n\n### 2. Concerts in New York Tomorrow\n\nHere are some resources and notable concerts happening in New York City tomorrow. For the most up-to-date listings, showtimes, and ticket availability, it\'s best to check the venue or ticketing websites directly.\n\n**How to Find Shows:**\n\n*   **Ticketmaster** and **Live Nation:** For major arena and theater shows.\n*   **Songkick** or **Bandsintown:** Excellent apps for tracking artists and finding concerts at venues of all sizes.\n*   **Venue Websites:** Check the calendars for specific venues like Madison Square Garde

---

## Summary

In this notebook you completed the following steps:

1. ✅ **Connected to AI Core** — loaded credentials and initialized `GenAIHubProxyClient`
2. ✅ **Configured parameters** — dataset path, prompt name, models, and metric
3. ✅ **Loaded and normalized** the BFCL v3 dataset — robust multi-format reader, tool normalization, golden record builder
4. ✅ **Uploaded 4 files** to AI Core dataset storage — train, test, tools, prompt template
5. ✅ **Registered a dataset artifact** — shared folder linked to `genai-optimizations` scenario
6. ✅ **Pushed base prompt template** to the Prompt Registry — `bfcl-tool-base:0.0.1`
7. ✅ **Created optimization configuration** — 15 parameter bindings, artifact binding
8. ✅ **Triggered and monitored** the optimization execution — polled until `COMPLETED`
9. ✅ **Retrieved the optimized prompt** — `bfcl-tool-optimized-gemini25:0.0.1` from the registry
10. ✅ **Compared base vs optimized** via live inference — confirmed optimization WIN

The optimized prompt `bfcl-tool-optimized-gemini25:0.0.1` is now registered in the Prompt Registry and ready to use for tool-calling inference with Gemini 2.5 Pro on SAP AI Core.